Pull a selection of MPC observations and up to three orbits and save the data locally.

To access MPC data you will need a Gcloud project with subscription to 2 BigQuery sources:
Main MPC Dataset (Analytics Hub listing) and Clustered Views Dataset (Analytics Hub listing).
Here we assume that subscription ids were the defaults, and the project id is provided as
an enviroment variable.

There is also an option to pull MPC data from local cache files, if you have them.

After this notebook is run, `data/orbit_fit_eval` will contain 4 Parquet files with
MPC observations, MPC orbits, SBDB orbits, and NEOCC orbits.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import List

import os
import pyarrow as pa
import pyarrow.compute as pc
from adam_core.orbits.query import query_neocc, query_sbdb_new
from adam_core.time import Timestamp
from adam_orbit_det_eval.utils import get_spacebased_stns
from mpcq import MPCObservations
from mpcq.client import BigQueryMPCClient
from mpcq.orbits import MPCOrbits
from pathlib import Path

In [ ]:
# If True, set local_stash_path below.
# Otherwise assume MPCQ_PROJECT_ID env variable has the name of the GCloud project
# with BigQuery subscription to the two datasets.
fetch_from_local_cache = True
local_stash_path = Path("../../data/actual/path")

In [ ]:
def use_mpc_query(object_ids: List):
    client = BigQueryMPCClient(
        dataset_id="asteroid_institute_mpc_replica",
        views_dataset_id="asteroid_institute___mpc_replica_views",
        project=os.environ["MPCQ_PROJECT_ID"],
    )
    observations = client.query_observations(object_ids)
    mpc_orbits = client.query_orbits(object_ids)
    return observations, mpc_orbits


def use_local_stash(object_ids: List, local_stash_dir: Path):
    observations = MPCObservations.from_parquet(
        local_stash_dir / "mpcq_observations.parquet"
    )
    observations = observations.apply_mask(
        pc.is_in(observations.requested_provid, pa.array(object_ids))
    )
    mpc_orbits = MPCOrbits.from_parquet(local_stash_dir / "mpcq_orbits.parquet")
    mpc_orbits = mpc_orbits.apply_mask(
        pc.is_in(mpc_orbits.requested_provid, pa.array(object_ids))
    )
    return observations, mpc_orbits


def clean_observations(observations):
    # Remove all space-based observations
    space_stns = get_spacebased_stns()
    observations = observations.apply_mask(
        pc.invert(pc.is_in(observations.stn, pa.array(space_stns)))
    )
    # Remove entries before we have data for ITRF93
    too_early_for_itrf93 = Timestamp.from_iso8601(["1961-06-01"])
    observations = observations.apply_mask(
        pc.greater(observations.obstime.mjd(), too_early_for_itrf93.mjd()[0])
    )
    # Remove dups
    observations = observations.sort_by(
        [
            ("obstime", "ascending"),
            ("requested_provid", "ascending"),
            ("stn", "ascending"),
        ]
    )
    mask = [True] * len(observations)
    for i in range(len(observations) - 1):
        mask[i] = (
            observations.obstime[i + 1] != observations.obstime[i]
            or observations.stn[i + 1] != observations.stn[i]
            or observations.requested_provid[i + 1] != observations.requested_provid[i]
        )
    observations = observations.apply_mask(mask)
    return observations


# The default path is relative to this notebook's location
def fetch_and_save(object_ids: List, out_dir: Path = Path("../../data/orbit_fit_eval")):
    out_dir.mkdir(parents=True, exist_ok=True)

    if fetch_from_local_cache:
        observations, mpc_orbits = use_local_stash(object_ids, local_stash_path)
    else:
        observations, mpc_orbits = use_mpc_query(object_ids)
    print(f"Before cleaning {len(observations)} observations")
    observations = clean_observations(observations)
    print(f"After cleaning {len(observations)} observations")

    observations.to_parquet(out_dir / "mpc_observations.parquet")
    mpc_orbits.to_parquet(out_dir / "mpc_orbits.parquet")

    sbdb_orbits = query_sbdb_new(object_ids)
    sbdb_orbits.to_parquet(out_dir / "sbdb_orbits.parquet")

    neocc_orbits = query_neocc(object_ids, orbit_type="ke", orbit_epoch="present-day")
    neocc_orbits.to_parquet(out_dir / "neocc_orbits.parquet")

In [ ]:
# Objects selected to cover a range of dynamical types, arc lengths, station coverage
desired_object_ids = [
    "2012 KJ25",
    "2007 CM57",
    "C/2023 U1",
    "2022 AC7",
    "1416 T-2",
    "1937 UD",
    "2019 LF6",
    "2021 PF20",
    "2002 CE26",
    "2002 EP58",
    "2003 RD6",
    "2007 BG",
    "2010 VE21",
    "2023 CL3",
    "2002 TP69",
    "2023 HU3",
    "2009 JY22",
    # these two blow up rchi2 for both MPC and SBDB, keeping them for experiments
    "1977 TJ3",
    "2009 NA",
    # HYA really blow up max rchi2 for SBDB, but not MPC
    "C/2022 E2",
    "C/2021 G2",
]

In [ ]:
fetch_and_save(desired_object_ids)